In [57]:
from os import PathLike
from hdfs import InsecureClient
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from delta import *
from pyspark.sql.types import LongType, StringType, StructField, StructType, BooleanType, ArrayType, IntegerType, DoubleType

In [58]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Python Spark SQL Hive integration for the EDSTD project") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [59]:
spark.sql(
    """
    SHOW TABLES FROM projeto
    """
).show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|  projeto|     amazon_books|      false|
|  projeto|   amazon_credits|      false|
|  projeto|    amazon_titles|      false|
|  projeto| book_movie_adapt|      false|
|  projeto|    book_to_movie|      false|
|  projeto|books_movie_adapt|      false|
|  projeto|  netflix_credits|      false|
|  projeto|   netflix_titles|      false|
+---------+-----------------+-----------+



In [60]:
delta_lake_path = "hdfs://hdfs-nn:9000/warehouse/projeto.db/streaming_titles/"

In [61]:
spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS projeto.streaming_titles (
        id VARCHAR(100),
        title VARCHAR(130),
        type VARCHAR(5),
        description VARCHAR(2200),
        age_certification VARCHAR(5),
        runtime INT,
        genres VARCHAR(105),
        production_countries VARCHAR(50),
        seasons DOUBLE,
        imdb_id VARCHAR(10),
        imdb_score DOUBLE,
        imdb_votes DOUBLE,
        tmdb_popularity DOUBLE,
        tmdb_score DOUBLE,
        platform VARCHAR(10)
    )
    USING DELTA
    PARTITIONED BY (release_year INT)
    LOCATION 'hdfs://hdfs-nn:9000/warehouse/projeto.db/streaming_titles/'
""")

DataFrame[]

In [62]:
spark.sql("""
    INSERT OVERWRITE TABLE projeto.streaming_titles
    PARTITION (release_year)
    
    SELECT 
        id, title, type, description, 
        age_certification, runtime, genres, production_countries, 
        seasons, imdb_id, imdb_score, imdb_votes, 
        tmdb_popularity, tmdb_score,
        'netflix' as platform,
        release_year
    FROM projeto.netflix_titles
    
    UNION ALL
    
    SELECT 
        id, title, type, description, 
        age_certification, runtime, genres, production_countries, 
        seasons, imdb_id, imdb_score, imdb_votes, 
        tmdb_popularity, tmdb_score,
        'amazon' as platform,
        release_year
    FROM projeto.amazon_titles
""")

DataFrame[]

In [63]:
spark.sql("""
    SELECT platform, COUNT(*) as total_count
    FROM projeto.streaming_titles
    GROUP BY platform
""").show()

+--------+-----------+
|platform|total_count|
+--------+-----------+
|  amazon|       9871|
| netflix|       5850|
+--------+-----------+



In [64]:
spark.sql("""
    SELECT * FROM projeto.streaming_titles
    LIMIT 20
""").show()


Amostra dos dados da tabela combinada:
+--------+--------------------+-----+--------------------+-----------------+-------+--------------------+--------------------+-------+---------+----------+----------+---------------+----------+--------+------------+
|      id|               title| type|         description|age_certification|runtime|              genres|production_countries|seasons|  imdb_id|imdb_score|imdb_votes|tmdb_popularity|tmdb_score|platform|release_year|
+--------+--------------------+-----+--------------------+-----------------+-------+--------------------+--------------------+-------+---------+----------+----------+---------------+----------+--------+------------+
| tm12280|    Harold and Maude|MOVIE|The young Harold ...|               PG|     91|['drama', 'comedy...|              ['US']|   null|tt0067185|       7.9|   76554.0|         11.998|       7.7|  amazon|        1971|
| ts21930|     Lupin the Third| SHOW|Arsene Lupin III ...|            TV-14|     23|['scifi', 'a

In [65]:
spark.stop()